# 实验 7：现成模型手势识别与固定机械臂动作

本 notebook 独立于实验 5 的四色识别。手势映射：**One→选择张开夹爪；Two→选择闭合夹爪；Three→选择回到零点；OK→确认执行；Five→停止锁定。** One、Two、Five 使用官方预置类别；Three 和 OK 使用官方手部关键点及明确的几何规则。几何余量分数不是概率。选择和确认各需连续 3 帧达到 0.70 识别分数，五指张开需连续 2 帧。每次执行后必须手动点击“准备下一条指令”，避免保持手势时反复执行。默认演练模式；现场核对机械臂周围无人、夹爪没有障碍物后，再点击“切换为真机模式”。五指张开停止只阻止后续动作，无法中断已经发送给现有 SDK 的阻塞运动。

模型和安装来源见 [https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer/python](https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer/python)。

In [ ]:
import time
from pathlib import Path
import cv2
import numpy as np
import mediapipe as mp
import math

目录 = Path('/home/jetson/Notebook/5.目标识别')
模型路径 = 目录 / 'models/gesture_recognizer.task'
相机号 = 0
最低置信度 = 0.70
手势中文 = {'One': '食指竖起', 'Two': '食指和中指竖起', 'Three': '食指、中指和无名指竖起', 'OK': '拇指与食指成环', 'Five': '五指张开'}
有效手势 = tuple(手势中文)
伸直阈值 = 1.85
弯曲阈值 = 1.55
OK距离阈值 = 0.42

def 创建识别器(模式):
    """用已下载的官方模型创建 MediaPipe 识别器。"""
    if not 模型路径.is_file():
        raise FileNotFoundError('缺少手势模型：' + str(模型路径))
    选项 = mp.tasks.vision.GestureRecognizerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(模型路径)),
        running_mode=模式,
        num_hands=1,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp.tasks.vision.GestureRecognizer.create_from_options(选项)

def 识别画面(图像, 识别器, 时间戳=None):
    """用官方类别识别一指、二指和五指张开，用关键点判断三指与 OK。"""
    彩色 = cv2.cvtColor(图像, cv2.COLOR_BGR2RGB)
    输入图 = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(彩色))
    if 时间戳 is None:
        原始结果 = 识别器.recognize(输入图)
    else:
        原始结果 = 识别器.recognize_for_video(输入图, 时间戳)
    官方类别, 官方分数 = '无手', 0.0
    if 原始结果.gestures and 原始结果.gestures[0]:
        首位 = 原始结果.gestures[0][0]
        官方类别, 官方分数 = 首位.category_name, float(首位.score)
    类别, 分数, 分数来源 = '未定义手势', 0.0, '无'
    if not 原始结果.hand_landmarks:
        类别 = '无手'
    else:
        点 = 原始结果.hand_landmarks[0]
        宽高比 = 图像.shape[1] / 图像.shape[0]
        def 距离(甲, 乙):
            return math.hypot((点[甲].x - 点[乙].x) * 宽高比, 点[甲].y - 点[乙].y)
        比值 = [距离(尖端, 根部) / max(距离(近端, 根部), 1e-6) for 根部, 近端, 尖端 in ((5,6,8),(9,10,12),(13,14,16),(17,18,20))]
        环距 = 距离(4, 8) / max(距离(5, 17), 1e-6)
        if 环距 <= OK距离阈值 and all(比值[i] >= 伸直阈值 for i in (1,2,3)):
            余量 = min(OK距离阈值-环距, *(比值[i]-伸直阈值 for i in (1,2,3)))
            类别, 分数, 分数来源 = 'OK', min(0.99, 0.70 + max(0.0, 余量)), '关键点几何余量'
        elif all(比值[i] >= 伸直阈值 for i in (0,1,2)) and 比值[3] <= 弯曲阈值:
            余量 = min(*(比值[i]-伸直阈值 for i in (0,1,2)), 弯曲阈值-比值[3])
            类别, 分数, 分数来源 = 'Three', min(0.99, 0.70 + max(0.0, 余量)), '关键点几何余量'
        elif 官方类别 in ('Pointing_Up', 'Victory', 'Open_Palm') and (官方类别 != 'Open_Palm' or 环距 > OK距离阈值):
            类别 = {'Pointing_Up':'One', 'Victory':'Two', 'Open_Palm':'Five'}[官方类别]
            分数, 分数来源 = 官方分数, '模型分类分数'
    通过 = 类别 in 有效手势 and 分数 >= 最低置信度 and len(原始结果.hand_landmarks) == 1
    return {'类别': 类别, '置信度': 分数, '通过': 通过, '分数来源': 分数来源, '官方类别': 官方类别, '官方分数': 官方分数, '关键点': 原始结果.hand_landmarks}

def 标注画面(图像, 结果):
    """在真实画面上绘制模型手部关键点和类别。"""
    标注 = 图像.copy()
    高, 宽 = 标注.shape[:2]
    for 手 in 结果['关键点']:
        for 点序号甲, 点序号乙 in mp.solutions.hands.HAND_CONNECTIONS:
            甲, 乙 = 手[点序号甲], 手[点序号乙]
            cv2.line(标注, (int(甲.x*宽), int(甲.y*高)), (int(乙.x*宽), int(乙.y*高)), (0, 255, 0), 2)
        for 点 in 手:
            cv2.circle(标注, (int(点.x*宽), int(点.y*高)), 3, (0, 0, 255), -1)
    文字 = '{} {:.2f}'.format(结果['类别'], 结果['置信度'])
    cv2.putText(标注, 文字, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return 标注

print('模型文件已找到：{}；可识别手势：{}'.format(模型路径.name, '、'.join(手势中文[x] for x in 有效手势)))


In [ ]:
from collections import deque
import threading

连续确认帧数 = 3
五指张开停止帧数 = 2
候选超时秒 = 8.0
动作映射 = {
    'One': '张开夹爪',
    'Two': '闭合夹爪',
    'Three': '回到零点',
}
确认手势 = 'OK'
停止手势 = 'Five'

class 手势状态机:
    """连续帧选动作、张掌确认、五指张开锁定停止，执行期间互锁。"""
    def __init__(self):
        self.状态 = 'WAIT'
        self.已准备 = False
        self.忙碌 = False
        self.停止锁定 = False
        self.候选 = None
        self.候选时间 = 0.0
        self.历史 = deque(maxlen=连续确认帧数)
        self.五指张开计数 = 0
        self.上次执行手势 = None

    def 准备(self):
        if self.忙碌 or self.停止锁定:
            return False
        self.状态, self.已准备, self.候选 = 'WAIT', True, None
        self.历史.clear()
        return True

    def 停止(self):
        self.停止锁定 = True
        self.已准备 = False
        self.候选 = None
        self.历史.clear()
        self.状态 = 'STOPPED'

    def 解除停止(self):
        if self.忙碌:
            return False
        self.停止锁定 = False
        self.状态 = 'WAIT'
        self.五指张开计数 = 0
        self.历史.clear()
        return True

    def 输入(self, 类别, 分数, 时间=None):
        """只返回一次待执行动作；五指张开优先进入停止锁定。"""
        时间 = time.monotonic() if 时间 is None else 时间
        if self.停止锁定:
            return None
        有效 = 类别 if 分数 >= 最低置信度 else None
        self.五指张开计数 = self.五指张开计数 + 1 if 有效 == 停止手势 else 0
        if self.五指张开计数 >= 五指张开停止帧数:
            self.停止()
            return '停止锁定'
        if not self.已准备 or self.忙碌 or self.停止锁定:
            return None
        if self.状态 == 'SELECT' and 时间 - self.候选时间 > 候选超时秒:
            self.状态, self.候选 = 'WAIT', None
            self.历史.clear()
        允许 = 有效 if 有效 in (*动作映射, 确认手势) else None
        self.历史.append(允许)
        if len(self.历史) < 连续确认帧数 or len(set(self.历史)) != 1 or 允许 is None:
            return None
        self.历史.clear()
        if self.状态 == 'WAIT' and 允许 in 动作映射:
            self.状态, self.候选, self.候选时间 = 'SELECT', 允许, 时间
            return None
        if self.状态 == 'SELECT' and 允许 == 确认手势:
            self.状态 = 'CONFIRM'
            动作 = 动作映射[self.候选]
            self.上次执行手势 = self.候选
            self.状态, self.忙碌, self.已准备 = 'EXECUTE', True, False
            return 动作
        if self.状态 == 'SELECT' and 允许 in 动作映射:
            self.候选, self.候选时间 = 允许, 时间
        return None

    def 执行结束(self):
        self.忙碌 = False
        self.状态 = 'STOPPED' if self.停止锁定 else 'WAIT'
        self.候选 = None

状态机 = 手势状态机()
print('已设定选择、确认、停止与执行互锁。')


## 离线状态机自检（不调用摄像头或机械臂）

In [ ]:
演练机 = 手势状态机()
assert 演练机.准备()
for _ in range(连续确认帧数):
    assert 演练机.输入('One', 0.90, 1.0) is None
assert 演练机.状态 == 'SELECT'
for _ in range(连续确认帧数):
    动作 = 演练机.输入('OK', 0.90, 2.0)
assert 动作 == '张开夹爪' and 演练机.忙碌
assert 演练机.输入('Two', 0.99, 3.0) is None
演练机.执行结束()
assert not 演练机.已准备
演练机.准备()
assert 演练机.输入('Five', 0.90, 4.0) is None
assert 演练机.输入('Five', 0.90, 5.0) == '停止锁定'
assert not 演练机.准备()
assert 演练机.解除停止()
print('离线自检通过：连续确认、忙碌互锁、五指张开停止、解除停止。')


In [ ]:
import sys

真实机械臂模式 = False
机械臂 = None
机械臂目录 = Path('/home/jetson/Notebook/4.机械臂OpenCV视觉处理实验')

def 连接机械臂():
    """手动连接现有 SDK；初始化时不自动移动机械臂。"""
    global 机械臂
    if 机械臂 is None:
        sys.path.insert(0, str(机械臂目录))
        from arm5dof_uservo import Arm5DoFUServo
        机械臂 = Arm5DoFUServo(is_init_pose=False, config_folder=str(机械臂目录 / 'config'))
    return 机械臂

def 执行固定动作(动作):
    """演练模式仅记录动作；真机模式执行现有 SDK 的固定动作。"""
    if not 真实机械臂模式:
        return '演练：' + 动作
    设备 = 连接机械臂()
    if 动作 == '张开夹爪':
        设备.gripper_open()
    elif 动作 == '闭合夹爪':
        设备.gripper_close()
    elif 动作 == '回到零点':
        设备.home()
    else:
        raise ValueError('未知动作：' + 动作)
    return '真机：' + 动作


## 实操顺序

先启动实时识别，观察画面中的 One、Two、Three、OK、Five 标签。点击“准备下一条指令”，做 One、Two 或 Three，稳定后放松，再做 OK 确认。在演练模式核对至少 10 轮随机手势、记录误触发和 Five 停止。需要真机验证时点击“切换为真机模式”，先单独测试张开夹爪，再测闭合夹爪和回到零点。每轮结束点击“记录真实测试”；全部结束点击“停止并释放摄像头”。

In [ ]:
import csv
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display

证据目录 = 目录 / 'gesture_evidence'
记录路径 = 目录 / 'week7_gesture_records.csv'
停止事件 = threading.Event()
取帧线程 = None
最近原图 = None
最近结果 = None
最近动作 = '未触发'
本轮已记录 = False

启动按钮 = widgets.Button(description='启动实时识别')
停止按钮 = widgets.Button(description='停止并释放摄像头')
准备按钮 = widgets.Button(description='准备下一条指令')
解除按钮 = widgets.Button(description='解除停止锁定')
真机按钮 = widgets.Button(description='切换为真机模式')
演练按钮 = widgets.Button(description='切换为演练模式')
截图按钮 = widgets.Button(description='保存截图')
记录按钮 = widgets.Button(description='记录真实测试')
预期框 = widgets.Dropdown(options=[(手势中文[x] + '→' + 动作映射[x], x) for x in 动作映射] + [('无动作', '无动作')], description='预期指令')
备注框 = widgets.Text(description='现场备注')
状态框 = widgets.HTML(value='等待启动；当前是演练模式。')
画面框 = widgets.Image(format='jpeg', width=640)

def 执行线程(动作):
    global 最近动作
    try:
        最近动作 = 执行固定动作(动作)
        状态框.value = 最近动作 + '；再次执行前请点击准备。'
    except Exception as 错误:
        最近动作 = '执行失败：' + str(错误)
        状态框.value = 最近动作
    finally:
        状态机.执行结束()

def 取帧循环():
    global 最近原图, 最近结果, 最近动作
    摄像头 = cv2.VideoCapture(相机号)
    识别器 = None
    try:
        if not 摄像头.isOpened():
            raise RuntimeError('摄像头无法打开；请检查其他 notebook 是否占用。')
        识别器 = 创建识别器(mp.tasks.vision.RunningMode.VIDEO)
        上次时间戳 = -1
        while not 停止事件.is_set():
            成功, 图像 = 摄像头.read()
            if not 成功:
                raise RuntimeError('摄像头取帧失败；实时循环已结束。')
            时间戳 = max(int(time.monotonic()*1000), 上次时间戳+1)
            上次时间戳 = 时间戳
            结果 = 识别画面(图像, 识别器, 时间戳)
            最近原图, 最近结果 = 图像.copy(), 结果
            事件 = 状态机.输入(结果['类别'], 结果['置信度'])
            if 事件 == '停止锁定':
                最近动作 = '五指张开停止锁定'
            elif 事件 in 动作映射.values():
                threading.Thread(target=执行线程, args=(事件,), daemon=True).start()
            成功编码, 编码 = cv2.imencode('.jpg', 标注画面(图像, 结果))
            if 成功编码:
                画面框.value = 编码.tobytes()
            状态框.value = '状态：{}；候选：{}；识别：{} {:.2f}；动作：{}；模式：{}'.format(状态机.状态, 手势中文.get(状态机.候选, '无'), 手势中文.get(结果['类别'], 结果['类别']), 结果['置信度'], 最近动作, '真机' if 真实机械臂模式 else '演练')
            停止事件.wait(0.12)
    except Exception as 错误:
        状态框.value = '实时识别出错：' + str(错误)
    finally:
        if 识别器 is not None:
            识别器.close()
        摄像头.release()
        状态机.停止()

def 启动(_):
    global 取帧线程
    if 取帧线程 is not None and 取帧线程.is_alive():
        状态框.value = '识别循环已经运行。'
        return
    停止事件.clear()
    取帧线程 = threading.Thread(target=取帧循环, daemon=True)
    取帧线程.start()

def 停止(_):
    状态机.停止()
    停止事件.set()
    状态框.value = '已停止接受指令；当前阻塞动作不能由此按钮紧急中断。摄像头稍后释放。'

def 准备(_):
    global 最近动作, 本轮已记录
    if 状态机.准备():
        最近动作, 本轮已记录 = '未触发', False
        状态框.value = '已准备；做 One、Two 或 Three，稍停，再做 OK 确认。'
    else:
        状态框.value = '忙碌或停止锁定中；不能准备下一条。'

def 解除(_):
    if 状态机.解除停止():
        状态框.value = '停止锁定已解除；点击准备按钮后才能再发指令。'
    else:
        状态框.value = '动作尚未结束；请等待。'

def 切换真机(_):
    global 真实机械臂模式
    if 状态机.忙碌:
        状态框.value = '动作执行中，不能切换模式。'
        return
    try:
        连接机械臂()
        真实机械臂模式 = True
        状态框.value = '真机模式已连接；初始化未发送运动指令。'
    except Exception as 错误:
        状态框.value = '真机连接失败：' + str(错误)

def 切换演练(_):
    global 真实机械臂模式
    if 状态机.忙碌:
        状态框.value = '动作执行中，不能切换模式。'
        return
    真实机械臂模式 = False
    状态框.value = '已切回演练模式。'

def 截图(_):
    if 最近原图 is None:
        状态框.value = '尚无真实画面。'
        return
    证据目录.mkdir(exist_ok=True)
    名称 = datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '_week7.png'
    cv2.imwrite(str(证据目录 / 名称), 标注画面(最近原图, 最近结果))
    状态框.value = '真实截图已保存：' + 名称

def 记录(_):
    global 本轮已记录
    if 最近原图 is None:
        状态框.value = '请先运行真实摄像头测试。'
        return
    if 状态机.忙碌:
        状态框.value = '动作尚未结束，稍后再记录。'
        return
    if 本轮已记录:
        状态框.value = '本轮已经记录；下一轮请重新点击准备。'
        return
    证据目录.mkdir(exist_ok=True)
    时间 = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    图片名 = 时间 + '_week7_trial.png'
    cv2.imwrite(str(证据目录 / 图片名), 标注画面(最近原图, 最近结果))
    预期 = 动作映射.get(预期框.value, '无动作')
    正确 = 最近动作 == '未触发' if 预期 == '无动作' else 最近动作 in ('演练：'+预期, '真机：'+预期)
    新文件 = not 记录路径.exists()
    with 记录路径.open('a', newline='', encoding='utf-8-sig') as 文件:
        表格 = csv.writer(文件)
        if 新文件:
            表格.writerow(['时间', '实际选择手势', '预期动作', '最后识别手势', '识别分数', '实际动作', '是否正确', '模式', '备注', '截图'])
        表格.writerow([时间, 预期框.value, 预期, 最近结果['类别'], '{:.4f}'.format(最近结果['置信度']), 最近动作, int(正确), '真机' if 真实机械臂模式 else '演练', 备注框.value, 图片名])
    本轮已记录 = True
    状态框.value = '本轮真实测试已记录；下一轮请重新准备。'

启动按钮.on_click(启动)
停止按钮.on_click(停止)
准备按钮.on_click(准备)
解除按钮.on_click(解除)
真机按钮.on_click(切换真机)
演练按钮.on_click(切换演练)
截图按钮.on_click(截图)
记录按钮.on_click(记录)
display(widgets.VBox([widgets.HBox([启动按钮, 停止按钮, 准备按钮, 解除按钮]), widgets.HBox([真机按钮, 演练按钮, 截图按钮]), 画面框, widgets.HBox([预期框, 备注框, 记录按钮]), 状态框]))
